# M25 · Contextual bandits

_Curriculum · Domain 5 · Reinforcement learning_

**Choose creative variants while learning which one works.**

We compare epsilon-greedy, UCB, and Thompson sampling on a deterministic Bernoulli-bandit simulation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(25)

## Bandit objective

A bandit chooses an action, observes only that action's reward, and repeats. Regret compares your reward with the best arm in hindsight:

$$R_T = T\mu^* - \sum_{t=1}^T r_t.$$

In [ ]:
true_ctr = np.array([0.035, 0.045, 0.055])
arm_names = np.array(["benefit", "proof", "urgency"])
rounds = 600

print(pd.DataFrame({"arm": arm_names, "true_ctr": true_ctr}))
assert true_ctr.argmax() == 2

## Epsilon-greedy

Most rounds exploit the current best empirical mean; a small fraction explores a random arm.

In [ ]:
def run_epsilon(epsilon):
    counts = np.zeros(3)
    wins = np.zeros(3)
    rewards = []
    choices = []
    for t in range(rounds):
        if rng.random() < epsilon or counts.sum() == 0:
            arm = rng.integers(3)
        else:
            means = wins / np.maximum(counts, 1)
            arm = int(np.argmax(means))
        reward = float(rng.random() < true_ctr[arm])
        counts[arm] = counts[arm] + 1
        wins[arm] = wins[arm] + reward
        rewards.append(reward)
        choices.append(arm)
    return np.array(rewards), np.array(choices), counts

eps_rewards, eps_choices, eps_counts = run_epsilon(0.10)
print(eps_counts.astype(int))
assert eps_counts.sum() == rounds

## UCB

UCB adds optimism to arms with fewer observations:

$$UCB_a = \hat\mu_a + c\sqrt{\frac{\log t}{n_a}}.$$

That bonus is the exploration engine.

In [ ]:
def run_ucb(c):
    counts = np.ones(3)
    wins = np.array([float(rng.random() < p) for p in true_ctr])
    rewards = wins.tolist()
    choices = [0, 1, 2]
    for t in range(3, rounds):
        means = wins / counts
        bonus = c * np.sqrt(np.log(t + 1) / counts)
        arm = int(np.argmax(means + bonus))
        reward = float(rng.random() < true_ctr[arm])
        counts[arm] = counts[arm] + 1
        wins[arm] = wins[arm] + reward
        rewards.append(reward)
        choices.append(arm)
    return np.array(rewards), np.array(choices), counts

ucb_rewards, ucb_choices, ucb_counts = run_ucb(0.4)
print(ucb_counts.astype(int))
assert ucb_counts.sum() == rounds

## Thompson sampling

For Bernoulli rewards, a Beta posterior is convenient. Thompson samples one plausible CTR per arm and chooses the largest sample.

In [ ]:
def run_thompson():
    alpha = np.ones(3)
    beta = np.ones(3)
    rewards = []
    choices = []
    for t in range(rounds):
        samples = rng.beta(alpha, beta)
        arm = int(np.argmax(samples))
        reward = float(rng.random() < true_ctr[arm])
        alpha[arm] = alpha[arm] + reward
        beta[arm] = beta[arm] + 1.0 - reward
        rewards.append(reward)
        choices.append(arm)
    return np.array(rewards), np.array(choices), alpha, beta

ts_rewards, ts_choices, ts_alpha, ts_beta = run_thompson()
print(np.round(ts_alpha / (ts_alpha + ts_beta), 4))
assert len(ts_rewards) == rounds

## Regret comparison

Lower regret means the algorithm found the useful creative faster.

In [ ]:
best_rate = true_ctr.max()
eps_regret = np.cumsum(best_rate - eps_rewards)
ucb_regret = np.cumsum(best_rate - ucb_rewards)
ts_regret = np.cumsum(best_rate - ts_rewards)

print(round(eps_regret[-1], 2))
print(round(ucb_regret[-1], 2))
print(round(ts_regret[-1], 2))
assert np.isfinite(ts_regret[-1])

## Plot the learning curves

A single run is noisy by design, which is why bandit evaluations use many seeds. The mechanics are still visible in one small run.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(eps_regret, label="epsilon")
ax.plot(ucb_regret, label="ucb")
ax.plot(ts_regret, label="thompson")
ax.set_title("cumulative regret")
ax.set_xlabel("round")
ax.set_ylabel("regret")
ax.legend()
plt.show()